In [3]:
import pandas as pd
import numpy as np
import re
import os

In [4]:
df = pd.read_excel("Citas_Digital.xlsx")

print("Filas y columnas originales:", df.shape)
df.head()

Filas y columnas originales: (1048575, 16)


,BDC,Fecha,Nombre Cliente,Telefono,Estatus de Lead,Potencial de compra,PDM,SDC,Venta,Asesor Asignado,¿Por que no ha visitado la agencia?,¿Por que no hay solicitud de credito?,¿Por que no hubo prueba de manejo ?,¿Por que no hubo proceso wow?,¿Por que se asigno antes de vistar piso?,Unnamed: 15
0,Saul,NaT,Guillermo,NaN,Finalizado,0.9,1.0,0.0,0.0,Nancy,"Ya visito la Agencia el dia jueves 17, intere...","Pide arrendamiento, unicamente se esta en espe...","Si, hizo prueba de manejo","Si recibio procesos wow, estuvo en la agencia ...","No, se espero a que visitara la agencia para s...",Comenta que no se le hizo atractivo el arrenda...
1,Saul,NaT,Roberto,NaN,Finalizado,0.4,0.0,0.0,0.0,Victor,Ya visito la Agencia el dia 3 de abril,Por que primero queria manejar las unidad,No habia unidad GS8 HEV para prueba de manejo,"Si recibio procesos wow, sin embargo no quizo ...",Se asigno hasta que visitara Piso,NaN
2,Saul,NaT,Irving,NaN,Finalizado,0.85,0.0,0.0,0.0,Alan,Ya visito la agencia el dia 21 de abril,"Firmo solicitud, sin embargo despues comento q...",Por que no habia unidad para prueba de manejo,"Si se realizo el proceso wow, los clientes se ...",Se asigno hasta que visitara Piso,Se le manda msj de seguimiento 03/05/2025
3,Saul,NaT,Oswaldo,NaN,Venta,0.9,0.0,0.0,1.0,Mauricio,El cliente vive en Tabasco Villahermosa,Sera compra de contado,"Ya la manejo en Villahermosa, sin embargo por ...","Las veces que se le ha marcado, agradece mucho...","Necesitabamos sacar el apartado, por eso se as...",NaN
4,Saul,NaT,Rodrigo,NaN,Finalizado,0.9,1.0,0.0,0.0,Mauricio,"Visito la agencia el mes pasado, y nos visita ...",Hoy visita nuevamente la agencia y se firma so...,"La primera vez que nos visito, realizo PDM",Si recibio el proceso wow la primera vez que n...,Se asigno hasta que visitara Piso la primera ...,Por el detalle del carbilink compro en otra ma...


In [5]:
# Paso 1- Filtro por filas (0-673)
df = df.loc[0:673]

print("Filas después del filtro:", df.shape[0])

Filas después del filtro: 674


In [6]:
#Paso 2: Filtro por columnas (0,1,2,4,5,6,7,8,9)
columnas_deseadas = [0, 1, 2, 4, 5, 6, 7, 8, 9]
df = df.iloc[:, columnas_deseadas]

print("Columnas después del filtro:", list(df.columns))

Columnas después del filtro: ['BDC ', 'Fecha', 'Nombre Cliente ', 'Estatus de Lead', 'Potencial de compra', 'PDM', 'SDC', 'Venta', 'Asesor Asignado']


In [7]:
#Paso 3: Convertir "Potencial de compra" a float
def limpiar_potencial(valor):

    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)

    # Se queda solo con dígitos y punto decimal
    solo_numeros = re.sub(r"[^0-9.]", "", str(valor))

    if solo_numeros == "":
        return np.nan  # ej. 'Ventas ' no tiene ningún número

    numero = float(solo_numeros)

    # Si el número es mayor a 1, asumimos que viene en formato porcentaje
    if numero > 1:
        numero = numero / 100

    return numero

df["Potencial de compra"] = df["Potencial de compra"].apply(limpiar_potencial)
df["Potencial de compra"] = df["Potencial de compra"].astype(float)

print(df["Potencial de compra"].dtype)  # debe decir float64
df["Potencial de compra"].head(10)

float64


0    0.90
1    0.40
2    0.85
3    0.90
4    0.90
5    0.85
6    0.20
7    0.50
8    0.80
9    0.80
Name: Potencial de compra, dtype: float64

In [ ]:
#Paso 5: Dividir por mes (Abril 2025 a Julio 2026)
# Aseguramos que la columna Fecha sea tipo fecha
df["Fecha"] = pd.to_datetime(df["Fecha"])

# Diccionario para traducir el número de mes a nombre en español
meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

# Generamos la lista de los 16 periodos: Abril 2025 -> Julio 2026
periodos = pd.date_range(start="2025-04-01", end="2026-07-01", freq="MS")

for periodo in periodos:
    anio = periodo.year
    mes_num = periodo.month
    nombre_mes = meses[mes_num]

    # Filtra las filas que correspondan a ese mes y año
    df_mes = df[(df["Fecha"].dt.year == anio) & (df["Fecha"].dt.month == mes_num)]

    nombre_archivo = f"{nombre_mes}_{anio}.csv"
    ruta_completa = os.path.join(carpeta_salida, nombre_archivo)

    df_mes.to_csv(ruta_completa, index=False, encoding="utf-8-sig")
    print(f"{nombre_archivo}: {len(df_mes)} registros")

